In [1]:
import numpy as np
import seaborn as sns
import os
from scipy.stats import kurtosis, skew
import pandas as pd
import pickle

from rcv_distribution import *
from MDS_analysis import *
from voting_rules import *
from consistency import *
from null_elections import *

Dropping trucated ballots from the real eletions:
1. Only bullet votes
2. All voluntarily truncation 

In [2]:
# Building the Dataframe

directory = "dataverse_files_2025"
election_table = pd.read_csv("election_table.csv")

# Dataframe: truncation analysis
trunc = pd.DataFrame(columns = ["filename", "candidates", "choices", 
                                "gamma", "og_irv", "og_condorcet", 
                                "og_plurality", "Bvote_free_irv", 
                                "Bvote_free_condorcet", "Bvote_free_plurality",
                                "Tvote_free_irv", "Tvote_free_condorcet", "Tvote_free_plurality",
                                "bullet_votes", "voluntarily_truncated_votes", "total_votes"
                                ])


for filename in os.listdir(directory):
    if filename in election_table["filename"].values:
        candidates = election_table.loc[election_table["filename"]==filename, "candidates"].values[0]
        if candidates > 2:
            trunc.loc[len(trunc)] = [pd.NA] * len(trunc.columns)
            
            i = len(trunc) - 1
            trunc.at[i, "filename"] = filename

            
            trunc.at[i, "candidates"] = candidates
            trunc.at[i, "choices"] = election_table.loc[election_table["filename"]==filename, "choices"].values[0]
            trunc.at[i, "gamma"] = election_table.loc[election_table["filename"]==filename, "gamma"].values[0]
            

            trunc.at[i, "og_irv"]  = election_table.loc[election_table["filename"]==filename, "irv_winner"].values[0]
            trunc.at[i, "og_condorcet"]  = election_table.loc[election_table["filename"]==filename, "condorcet_winner"].values[0]
            trunc.at[i, "og_plurality"]  = election_table.loc[election_table["filename"]==filename, "plurality_winner"].values[0]



In [3]:
trunc

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# --- prepare data from `trunc` ---
trunc["gamma"] = pd.to_numeric(trunc["gamma"], errors="coerce")
trunc["candidates"] = pd.to_numeric(trunc["candidates"], errors="coerce")

dfv = trunc.loc[trunc["gamma"].notna() & trunc["candidates"].notna(), ["level", "candidates", "gamma"]].copy()
dfv["one_minus_gamma"] = 1 - dfv["gamma"]

# (same filter as before) keep only elections with < 16 candidates
dfv = dfv[dfv["candidates"] < 16]

# aggregate (mean of 1 - gamma) by level
order_raw   = ["LOCAL", "STATE", "FEDERAL"]
order_labels= ["Local", "State", "Federal"]

agg = (
    dfv.groupby("level", as_index=False)["one_minus_gamma"]
       .mean()
       .set_index("level")
       .reindex(order_raw)   # keep desired order; may produce NaN if a level is missing
       .dropna()             # drop levels not present
       .reset_index()
)

levels = [order_labels[order_raw.index(lv)] for lv in agg["level"]]
vals   = (agg["one_minus_gamma"] * 100).to_list()  # percent scale

x = list(range(len(levels)))  # 0,1,...

fig, ax = plt.subplots(figsize=(6.5, 4.2), dpi=150)

# single series plotted like your format
ax.scatter(x, vals, s=60, marker='o', label='1 − gamma (mean)', zorder=3)
ax.plot(x, vals, lw=0.8, alpha=0.6, zorder=2, solid_capstyle='round')

# ticks/axes
ax.set_xticks(x, levels)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_ylim(0, 100)
ax.grid(axis='y', linestyle=':', linewidth=0.8, alpha=0.6)

ax.set_xlabel("Level")
ax.set_ylabel("1 − gamma")

# annotate values above points
for xi, y in zip(x, vals):
    ax.annotate(f'{y:.0f}%', (xi, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)

plt.title("1 − gamma by level (candidates < 16)")
ax.legend(frameon=False, loc='best')
plt.tight_layout()
# plt.savefig('one_minus_gamma_by_level_lt16.png', dpi=300)

plt.show()


KeyError: "['level'] not in index"

In [6]:
def load_data(filename, save_folder):
    # Extract the base filename without extension
    base_filename = os.path.splitext(os.path.basename(filename))[0]

    # Create the save paths for dictionary and list
    dict_load_path = os.path.join(save_folder, f"{base_filename}_ballots.pkl")
    list_load_path = os.path.join(save_folder, f"{base_filename}_candidates.pkl")

    # Load the dictionary
    with open(dict_load_path, 'rb') as dict_file:
        data_dict = pickle.load(dict_file)

    # Load the list
    with open(list_load_path, 'rb') as list_file:
        data_list = pickle.load(list_file)

    return data_dict, data_list

In [9]:
# dropping the only bullet_vote  ballots and recalculating winners 
saved = "saved_ballots_and_candidates"
total_change = 0
total = 0
Bvotes_average = 0
for filename in os.listdir(directory):
    if filename in trunc["filename"].values:
        base_filename = filename[:-4]
        try:
            ballots, candidates = load_data(base_filename, saved)
        except Exception as e:
            print(filename, " ", e)

        altered_ballots = {}
        Bvotes = 0
        total_votes = 0
        for b in ballots:
            if len(b) > 0:
                total_votes += ballots[b]
            
            if len(b) > 1:
                altered_ballots[b] = ballots[b]
            elif len(b) == 1:
                Bvotes += ballots[b]

        Bvotes_average += (Bvotes / total_votes)
            
        election = voting_rules(altered_ballots, candidates)
        Bvote_free_irv = election.irv()[0]
        # print(Bvote_free_irv, " ", trunc.loc[trunc["filename"] == filename, "og_irv"].values[0], " ", Bvotes, "/", total_votes)
        trunc.loc[trunc["filename"]==filename, "Bvote_free_irv"] = Bvote_free_irv

        Bvote_free_condorcet = election.condorcet()
        trunc.loc[trunc["filename"]==filename, "Bvote_free_condorcet"] = Bvote_free_condorcet

        Bvote_free_plurality = election.plurality()
        trunc.loc[trunc["filename"]==filename, "Bvote_free_plurality"] = Bvote_free_plurality 

        trunc.loc[trunc["filename"]==filename, "bullet_votes"] = Bvotes
        trunc.loc[trunc["filename"]==filename, "total_votes"] = total_votes



        if (Bvote_free_irv != trunc.loc[trunc["filename"]==filename, "og_irv"].values[0]):
            total_change += 1
        total += 1
        
print(total_change, "/", total)
print(Bvotes_average/total)





53 / 361
0.3430334387926418


In [69]:
# dropping truncated ballots

saved = "saved_ballots_and_candidates"
total_change = 0
total = 0
wierd = 0
for filename in os.listdir(directory):
    if filename in trunc["filename"].values:
        base_filename = filename[:-4]
        try:
            ballots, candidates = load_data(base_filename, saved)
        except Exception as e:
            print(filename, " ", e)

        choices = trunc.loc[trunc["filename"]==filename, "choices"].values[0]
        candidates_num = trunc.loc[trunc["filename"]==filename, "candidates"].values[0]

        if choices >= candidates_num: 
            altered_ballots = {}
            
            Tvotes = 0
            flag = False
            for b in ballots:
                if len(b) == candidates_num:
                    altered_ballots[b] = ballots[b]
                elif len(b) > 0 and len(b) < candidates_num:
                    Tvotes += ballots[b]
                elif len(b) > candidates_num:
                    flag = True
            
            if flag is True:
                wierd += 1
                print(filename, " is wierd")
            
            election = voting_rules(altered_ballots, candidates)
            Tvote_free_irv = election.irv()[0]
            # print(Bvote_free_irv, " ", trunc.loc[trunc["filename"] == filename, "og_irv"].values[0], " ", Bvotes, "/", total_votes)
            trunc.loc[trunc["filename"]==filename, "Tvote_free_irv"] = Tvote_free_irv

            Tvote_free_condorcet = election.condorcet()
            trunc.loc[trunc["filename"]==filename, "Tvote_free_condorcet"] = Tvote_free_condorcet

            Tvote_free_plurality = election.plurality()
            trunc.loc[trunc["filename"]==filename, "Tvote_free_plurality"] = Tvote_free_plurality 

            trunc.loc[trunc["filename"]==filename, "voluntarily_truncated_votes"] = Tvotes


            if (Tvote_free_irv != Tvote_free_condorcet):
                total_change += 1
            else:
                total += 1
        
        
print(total_change, "/", total)





5 / 188


In [70]:
trunc

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,3922,<NA>,19742
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333,188852
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747,338650
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769,8163
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576,8820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385,1537
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452,1089
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663,9560
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262,655


In [72]:
trunc ["level"] = pd.NA

In [76]:
trunc_copy = trunc.copy()
trunc_copy

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,3922,<NA>,19742,<NA>
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333,188852,<NA>
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747,338650,<NA>
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769,8163,<NA>
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576,8820,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385,1537,<NA>
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452,1089,<NA>
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663,9560,<NA>
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262,655,<NA>


In [77]:
for i in range(len(trunc)):
    filename = trunc.at[i, "filename"]
    level = election_table.loc[election_table["filename"]==filename, "level"].values[0]
    trunc.at[i, "level"] = level

trunc

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,3922,<NA>,19742,FEDERAL
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333,188852,FEDERAL
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747,338650,FEDERAL
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769,8163,STATE
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576,8820,STATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385,1537,LOCAL
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452,1089,LOCAL
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663,9560,LOCAL
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262,655,LOCAL


In [78]:
valid = (
    trunc["Tvote_free_irv"].notna()
    & trunc["Tvote_free_condorcet"].notna()
    & trunc["candidates"].notna()
)
print(valid.value_counts().get(True, 0))
dfv = trunc.loc[valid, ["filename", "gamma", "level", "candidates", "Tvote_free_irv", "Tvote_free_condorcet"]].copy()
dfv["candidates"] = dfv["candidates"].astype(int)
dfv["differs"] = (dfv["Tvote_free_irv"] != dfv["Tvote_free_condorcet"]).astype(int)
print(dfv[dfv["differs"]==1])

193
                                           filename     gamma  level  \
3                  Alaska_11052024_StateHouseD1.csv  0.953816  STATE   
17   Alaska_11082022_GovernorLieutenantGovernor.csv  0.880671  STATE   
70                    Burlington_03032009_Mayor.csv  0.651282  LOCAL   
85      LasCruces_11052019_MAYORCITYOFLASCRUCES.csv   0.49374  LOCAL   
303      SanFrancisco_11052019_DistrictAttorney.csv  0.751599  LOCAL   

     candidates      Tvote_free_irv   Tvote_free_condorcet  differs  
3             3     Echohawk, Grant        Moran, Agnes C.        1  
17            4  Dunleavy/Dahlstrom          Walker/Drygas        1  
70            5            Bob Kiss          Andy Montroll        1  
85           10       MIKE A TELLEZ  WILLIAM BILL MATTIACE        1  
303           4        CHESA BOUDIN                     -1        1  


In [80]:
valid = (
    trunc["Bvote_free_irv"].notna()
    & trunc["Bvote_free_condorcet"].notna()
    & trunc["candidates"].notna()
)
dfv = trunc.loc[valid, ["filename", "gamma", "level", "candidates", "Bvote_free_irv", "Bvote_free_condorcet"]].copy()
dfv["candidates"] = dfv["candidates"].astype(int)
dfv["differs"] = (dfv["Bvote_free_irv"] != dfv["Bvote_free_condorcet"]).astype(int)
print(dfv[dfv["differs"]==1])


                                              filename     gamma  level  \
17      Alaska_11082022_GovernorLieutenantGovernor.csv  0.880671  STATE   
22                 Alaska_11082022_HouseDistrict20.csv  0.853639  STATE   
70                       Burlington_03032009_Mayor.csv  0.651282  LOCAL   
108                     Minneapolis_11022021_Mayor.csv  0.883745  LOCAL   
211          NewYorkCity_06222021_DEMMayorCitywide.csv  0.462661  LOCAL   
260  PierceCounty_11042008_CountyAssessorTreasurer.csv  0.823079  LOCAL   
287                    SanFrancisco_11032015_Mayor.csv  0.845836  LOCAL   

     candidates      Bvote_free_irv Bvote_free_condorcet  differs  
17            4  Dunleavy/Dahlstrom        Walker/Drygas        1  
22            4     Gray, Andrew T.                   -1        1  
70            5            Bob Kiss        Andy Montroll        1  
108          18       Sheila Nezhad           Kate Knuth        1  
211          13   Kathryn A. Garcia                   -1   

In [55]:
import numpy as np
import pandas as pd

# Ensure numeric
for col in ["bullet_votes", "voluntarily_truncated_votes", "total_votes"]:
    trunc[col] = pd.to_numeric(trunc[col], errors="coerce")

# Valid denominator
mask = trunc["total_votes"] > 0

# Ratios
bullet_rate = trunc.loc[mask, "bullet_votes"] / trunc.loc[mask, "total_votes"]
trunc_rate  = trunc.loc[mask, "voluntarily_truncated_votes"] / trunc.loc[mask, "total_votes"]

def stats(s: pd.Series) -> dict:
    s = s.dropna()
    n = int(s.size)
    mean = s.mean()
    median = s.median()
    sd = s.std(ddof=1) if n > 1 else np.nan      # sample SD
    se = (sd / np.sqrt(n)) if n > 1 else np.nan  # standard error of mean
    return {"n": n, "mean": mean, "median": median, "sd": sd, "se": se}

summary = pd.DataFrame({
    "bullet_votes/total_votes": stats(bullet_rate),
    "voluntarily_truncated_votes/total_votes": stats(trunc_rate),
}).T

print(summary)


                                             n      mean    median        sd  \
bullet_votes/total_votes                 361.0  0.343033  0.320828  0.142326   
voluntarily_truncated_votes/total_votes  193.0  0.642663  0.657517  0.175420   

                                               se  
bullet_votes/total_votes                 0.007491  
voluntarily_truncated_votes/total_votes  0.012627  


In [82]:
trunc.to_csv("truncation_analysis.csv")

In [84]:
import numpy as np
import pandas as pd

# Ensure numeric
for col in ["bullet_votes", "voluntarily_truncated_votes", "total_votes"]:
    trunc[col] = pd.to_numeric(trunc[col], errors="coerce")

# Valid rows
mask = trunc["total_votes"] > 0
df_rates = trunc.loc[mask, ["level"]].copy()
df_rates["bullet_rate"] = trunc.loc[mask, "bullet_votes"] / trunc.loc[mask, "total_votes"]
df_rates["trunc_rate"]  = trunc.loc[mask, "voluntarily_truncated_votes"] / trunc.loc[mask, "total_votes"]

# Order levels nicely (optional)
lvl_order = pd.CategoricalDtype(categories=["FEDERAL", "STATE", "LOCAL"], ordered=True)
df_rates["level"] = df_rates["level"].astype("string").astype(lvl_order)

def summarize(series: pd.Series) -> pd.DataFrame:
    out = series.agg(['count', 'mean', 'median', 'std']).to_frame().T
    out.rename(columns={'count':'n', 'std':'sd'}, inplace=True)
    out['se'] = out['sd'] / np.sqrt(out['n']).where(out['n'] > 1)
    return out[['n','mean','median','sd','se']]

# --- BY LEVEL ---
br = df_rates.groupby("level", observed=True)["bullet_rate"].apply(summarize)
tr = df_rates.groupby("level", observed=True)["trunc_rate"].apply(summarize)

# Flatten indexes and add prefixes to distinguish the two sets
br.columns = [f"bullet_{c}" for c in br.columns]
tr.columns = [f"trunc_{c}"  for c in tr.columns]
by_level = pd.concat([br, tr], axis=1).sort_index()

print("By level (fractions in [0,1]):")
print(by_level)

# --- OVERALL ---
overall_br = summarize(df_rates["bullet_rate"]).add_prefix("bullet_")
overall_tr = summarize(df_rates["trunc_rate"]).add_prefix("trunc_")
overall = pd.concat([overall_br, overall_tr], axis=1)
print("\nOverall summary:")
print(overall)


By level (fractions in [0,1]):
                 bullet_n  bullet_mean  bullet_median  bullet_sd  bullet_se  \
level                                                                         
FEDERAL FEDERAL      15.0     0.391689       0.359669   0.141341   0.036494   
STATE   STATE        42.0     0.493826       0.504081   0.168986   0.026075   
LOCAL   LOCAL       304.0     0.319799       0.300491   0.124377   0.007134   

                 trunc_n  trunc_mean  trunc_median  trunc_sd  trunc_se  
level                                                                   
FEDERAL FEDERAL     10.0    0.752484      0.779219  0.124988  0.039525  
STATE   STATE       42.0    0.724038      0.792366  0.177588  0.027402  
LOCAL   LOCAL      141.0    0.610635      0.608946  0.167572  0.014112  

Overall summary:
             bullet_n  bullet_mean  bullet_median  bullet_sd  bullet_se  \
bullet_rate     361.0     0.343033       0.320828   0.142326   0.007491   
trunc_rate        NaN          NaN      

Filling out Keena's Table 


In [10]:
ta = pd.read_csv("truncation_analysis.csv")
election_table = pd.read_csv("election_table.csv")

# Keena's table
df = pd.DataFrame()


# per-election bullet %
ta["%_bullet"] = 100 * ta["bullet_votes"] / ta["total_votes"]

# per-election truncated % (only where voluntarily_truncated_votes exists)
ta["%_truncated"] = np.where(
    ta["voluntarily_truncated_votes"].notna(),
    100 * ta["voluntarily_truncated_votes"] / ta["total_votes"],
    np.nan
)
ta

,Unnamed: 0,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level,%_bullet,%_truncated
0,0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,NaN,NaN,NaN,3922,NaN,19742,FEDERAL,19.866275,NaN
1,1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333.0,188852,FEDERAL,29.681444,70.601847
2,2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747.0,338650,FEDERAL,67.408239,89.102909
3,3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769.0,8163,STATE,65.049614,82.922945
4,4,Alaska_11052024_StateHouseD15.csv,3,4,0.982540,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576.0,8820,STATE,70.714286,85.895692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385.0,1537,LOCAL,20.624593,25.048796
357,357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452.0,1089,LOCAL,5.876951,41.505969
358,358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663.0,9560,LOCAL,46.694561,59.236402
359,359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262.0,655,LOCAL,19.541985,40.000000


counting the complete ballots

In [13]:
saved = "saved_ballots_and_candidates"
count = 0
# adding a "complete ballot" = all of the choices on the ballot are filled out 
# adding a "all candidates listed" = choices on the ballot = number of candidates 

ta["all_candidates_listed"] = np.nan
ta["all_choices_filled"] = np.nan

for filename in ta["filename"]:
    fn =  filename[:-4]+ ".pkl"
    ballots, candidates = load_data(fn, saved)
    if len(candidates) > 2:
        candidates, choices = ta.loc[ta["filename"]==filename, "candidates"].values[0], ta.loc[ta["filename"]==filename, "choices"].values[0]
        if candidates <= choices:
            # second condition
            complete_ballot_counter = 0
            for b in ballots:
                if len(b) == candidates:
                    complete_ballot_counter += ballots[b]
            ta.loc[ta["filename"]==filename , "all_candidates_listed"] = complete_ballot_counter
            

        if candidates > choices:
            # first condition
            complete_ballot_counter = 0
            for b in ballots:
                if len(b) == choices:
                    complete_ballot_counter += ballots[b]
            ta.loc[ta["filename"]==filename, "all_choices_filled"] = complete_ballot_counter
            
        ta.loc[ta["filename"]==filename, "complete_ballot (either case)"] = complete_ballot_counter
        ta["%_complete"] = 100 * ta["complete_ballot (either case)"] / ta["total_votes"]
        
ta




,Unnamed: 0,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,...,bullet_votes,voluntarily_truncated_votes,total_votes,level,%_bullet,%_truncated,all_candidates_listed,all_choices_filled,complete_ballot (either case),%_complete
0,0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,...,3922,NaN,19742,FEDERAL,19.866275,NaN,NaN,5344.0,5344.0,27.069193
1,1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick",...,56054,133333.0,188852,FEDERAL,29.681444,70.601847,55519.0,NaN,55519.0,29.398153
2,2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,...,228278,301747.0,338650,FEDERAL,67.408239,89.102909,36903.0,NaN,36903.0,10.897091
3,3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.",...,5310,6769.0,8163,STATE,65.049614,82.922945,1394.0,NaN,1394.0,17.077055
4,4,Alaska_11052024_StateHouseD15.csv,3,4,0.982540,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny",...,6237,7576.0,8820,STATE,70.714286,85.895692,1244.0,NaN,1244.0,14.104308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,...,317,385.0,1537,LOCAL,20.624593,25.048796,1152.0,NaN,1152.0,74.951204
357,357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,...,64,452.0,1089,LOCAL,5.876951,41.505969,637.0,NaN,637.0,58.494031
358,358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",...,4464,5663.0,9560,LOCAL,46.694561,59.236402,3897.0,NaN,3897.0,40.763598
359,359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,...,128,262.0,655,LOCAL,19.541985,40.000000,393.0,NaN,393.0,60.000000


In [35]:
cols = [
    "type of election", "N", "%_complete", "%_bullet", "%_truncated", "gamma"
]

df = pd.DataFrame(columns=cols)

df.loc[len(df)] = ["all elections"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["federal/state"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["local"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["partisan"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["non-partisan"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["primary"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["general"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["3 candidate races"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["4 candidate races"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["5 candidate races"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["6 candidate races"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["7 candidate races"] + [np.nan]*(len(cols)-1)
df.loc[len(df)] = ["8 candidate races"] + [np.nan]*(len(cols)-1)

df

,type of election,N,%_complete,%_bullet,%_truncated,gamma
0,all elections,NaN,NaN,NaN,NaN,NaN
1,federal/state,NaN,NaN,NaN,NaN,NaN
2,local,NaN,NaN,NaN,NaN,NaN
3,partisan,NaN,NaN,NaN,NaN,NaN
4,non-partisan,NaN,NaN,NaN,NaN,NaN
5,primary,NaN,NaN,NaN,NaN,NaN
6,general,NaN,NaN,NaN,NaN,NaN
7,3 candidate races,NaN,NaN,NaN,NaN,NaN
8,4 candidate races,NaN,NaN,NaN,NaN,NaN
9,5 candidate races,NaN,NaN,NaN,NaN,NaN


In [14]:
et = pd.read_csv("election_table.csv")

In [100]:
# ta column has either a number or the string "majority"
exh_raw = ta["%_exhausted"]

# boolean: is "majority"
is_majority = exh_raw.astype(str).str.strip().str.lower().eq("majority")

# numeric version (non-numbers -> NaN, including "majority")
exh_num = pd.to_numeric(exh_raw, errors="coerce")
m = ta.index == ta.index


In [52]:
df = pd.read_csv("summary_table.csv")
df

,type of election,N,%_complete,%_bullet,%_truncated,gamma,avg. candidates,%_exhausted,first_round_majority,complete_gamma
0,all elections,361,39.129348,34.303344,NaN,0.817405,5.573407,0.104294,132,0.604901
1,federal/state,57,27.760356,46.694809,NaN,0.884370,4.000000,0.074052,25,0.651544
2,local,304,41.261033,31.979944,NaN,0.804849,5.868421,0.109207,107,0.596097
3,partisan,52,27.760356,50.665682,NaN,0.905509,4.673077,0.089965,23,0.639910
4,non-partisan,244,43.852758,31.336091,NaN,0.826499,5.475410,0.098366,92,0.628307
5,primary,65,33.752729,32.352084,NaN,0.712783,6.661538,0.131723,17,0.489752
6,general,296,40.310024,34.731830,NaN,0.840379,5.334459,0.097020,115,0.630359
7,3 candidate races,125,39.010749,40.962466,NaN,0.899491,3.000000,0.061548,70,0.739125
8,4 candidate races,83,38.383081,36.136461,NaN,0.847514,4.000000,0.071464,33,0.474871
9,5 candidate races,42,39.741795,29.369791,NaN,0.759198,5.000000,0.088524,10,0.522661


In [63]:
# all elections
mask = pd.Series(True, index=ta.index)
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "all elections", "enough_choices"] = enough_mask.sum()


# federal/state
level = ta["filename"].map(et.set_index("filename")["level"])
mask = level.isin(["FEDERAL", "STATE"])
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "federal/state", "enough_choices"] = enough_mask.sum()


# local
level = ta["filename"].map(et.set_index("filename")["level"])
mask = level.isin(["LOCAL"])
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "local", "enough_choices"] = enough_mask.sum()


# partisan
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().eq("yes")
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "partisan", "enough_choices"] = enough_mask.sum()


# non-partisan
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().eq("no")
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "non-partisan", "enough_choices"] = enough_mask.sum()


# primary
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().isin(["rp", "dp"])
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "primary", "enough_choices"] = enough_mask.sum()


# general
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().isin(["yes", "no"])
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "general", "enough_choices"] = enough_mask.sum()


# 3 candidate races
mask = ta["candidates"] == 3
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "3 candidate races", "enough_choices"] = enough_mask.sum()


# 4 candidate races
mask = ta["candidates"] == 4
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "4 candidate races", "enough_choices"] = enough_mask.sum()


# 5 candidate races
mask = ta["candidates"] == 5
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "5 candidate races", "enough_choices"] = enough_mask.sum()


# 6 candidate races
mask = ta["candidates"] == 6
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "6 candidate races", "enough_choices"] = enough_mask.sum()


# 7 candidate races
mask = ta["candidates"] == 7
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "7 candidate races", "enough_choices"] = enough_mask.sum()


# 8 candidate races
mask = ta["candidates"] == 8
enough_mask = mask & (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"] == "8 candidate races", "enough_choices"] = enough_mask.sum()
df

,type of election,N,%_complete,%_bullet,%_truncated,gamma,avg. candidates,%_exhausted,first_round_majority,complete_gamma,enough_choices
0,all elections,361,0.366537,34.303344,NaN,0.817405,5.573407,0.104294,132,0.536824,255.0
1,federal/state,57,0.270491,46.694809,NaN,0.884370,4.000000,0.074052,25,0.637701,52.0
2,local,304,0.391262,31.979944,NaN,0.804849,5.868421,0.109207,107,0.510856,203.0
3,partisan,52,0.233626,50.665682,NaN,0.905509,4.673077,0.089965,23,0.630507,50.0
4,non-partisan,244,0.410157,31.336091,NaN,0.826499,5.475410,0.098366,92,0.546921,169.0
5,primary,65,0.347574,32.352084,NaN,0.712783,6.661538,0.131723,17,0.359591,36.0
6,general,296,0.369669,34.731830,NaN,0.840379,5.334459,0.097020,115,0.566092,219.0
7,3 candidate races,125,0.390107,40.962466,NaN,0.899491,3.000000,0.061548,70,0.739125,125.0
8,4 candidate races,83,0.383831,36.136461,NaN,0.847514,4.000000,0.071464,33,0.474871,83.0
9,5 candidate races,42,0.330493,29.369791,NaN,0.759198,5.000000,0.088524,10,0.177518,21.0


In [64]:
df.to_csv("summary_table.csv", index=False)

In [ ]:
# all elections:


df.loc[df["type of election"]=="all elections", "complete_gamma"] = ta["gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "all elections", "%_complete"] = (
    ta["complete_ballots_count"] / ta["total_votes"]
).mean(skipna=True)
mask =  (ta["candidates"] <= ta["choices"] + 1)
df.loc[df["type of election"]=="all elections", "enough_choices"] = mask.mean(skipna=True)

# federal/state
level = ta["filename"].map(et.set_index("filename")["level"])
mask = level.isin(["FEDERAL", "STATE"])

df.loc[df["type of election"]=="federal/state", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "federal/state", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# local
level = ta["filename"].map(et.set_index("filename")["level"])
mask = level.isin(["LOCAL"])
df.loc[df["type of election"]=="local", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "local", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)


# partisan
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().eq("yes")

df.loc[df["type of election"]=="partisan", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "partisan", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)


# non partisan
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().eq("no")

df.loc[df["type of election"]=="non-partisan", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "non-partisan", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)



# primary
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().isin(["rp", "dp"])

df.loc[df["type of election"]=="primary", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)

df.loc[df["type of election"] == "primary", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# general
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().isin(["yes", "no"])


df.loc[df["type of election"]=="general", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "general", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)


df = df.reset_index(drop=True)
df

,type of election,N,%_complete,%_bullet,%_truncated,gamma,avg. candidates,%_exhausted,first_round_majority,complete_gamma
0,all elections,361,0.366537,34.303344,NaN,0.817405,5.573407,0.104294,132,0.536824
1,federal/state,57,0.270491,46.694809,NaN,0.884370,4.000000,0.074052,25,0.637701
2,local,304,0.391262,31.979944,NaN,0.804849,5.868421,0.109207,107,0.510856
3,partisan,52,0.233626,50.665682,NaN,0.905509,4.673077,0.089965,23,0.630507
4,non-partisan,244,0.410157,31.336091,NaN,0.826499,5.475410,0.098366,92,0.546921
5,primary,65,0.347574,32.352084,NaN,0.712783,6.661538,0.131723,17,0.359591
6,general,296,0.369669,34.731830,NaN,0.840379,5.334459,0.097020,115,0.566092
7,3 candidate races,125,39.010749,40.962466,NaN,0.899491,3.000000,0.061548,70,0.739125
8,4 candidate races,83,38.383081,36.136461,NaN,0.847514,4.000000,0.071464,33,0.474871
9,5 candidate races,42,39.741795,29.369791,NaN,0.759198,5.000000,0.088524,10,0.177518


In [37]:
# all elections:

df.loc[df["type of election"]=="all elections", "N"] = 361
df.loc[df["type of election"]=="all elections", "%_complete"] = ta["%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="all elections", "%_bullet"] = ta["%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="all elections", "avg. candidates"] = ta["candidates"].mean(skipna=True)
df.loc[df["type of election"]=="all elections", "gamma"] = ta["gamma"].mean(skipna=True)
df.loc[df["type of election"]=="all elections", "%_exhausted"] = exh_num[m].mean(skipna=True)
df.loc[df["type of election"]=="all elections", "first_round_majority"] = is_majority[m].sum()
df.loc[df["type of election"]=="all elections", "complete_gamma"] = ta["gamma_complete"].mean(skipna=True)

# federal/state
level = ta["filename"].map(et.set_index("filename")["level"])
mask = level.isin(["FEDERAL", "STATE"])
mean_complete_fed_state = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="federal/state", "N"] = mask.sum()
df.loc[df["type of election"]=="federal/state", "%_complete"] = mean_complete_fed_state
df.loc[df["type of election"]=="federal/state", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="federal/state", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="federal/state", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)
df.loc[df["type of election"]=="federal/state", "%_exhausted"] = exh_num[mask].mean(skipna=True)
df.loc[df["type of election"]=="federal/state", "first_round_majority"] = is_majority[mask].sum()
df.loc[df["type of election"]=="federal/state", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)

# local
level = ta["filename"].map(et.set_index("filename")["level"])
mask = level.isin(["LOCAL"])
mean_complete_local = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="local", "N"] = mask.sum()
df.loc[df["type of election"]=="local", "%_complete"] = mean_complete_local
df.loc[df["type of election"]=="local", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="local", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="local", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)
df.loc[df["type of election"]=="local", "%_exhausted"] = exh_num[mask].mean(skipna=True)
df.loc[df["type of election"]=="local", "first_round_majority"] = is_majority[mask].sum()
df.loc[df["type of election"]=="local", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)

# partisan
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().eq("yes")

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="partisan", "N"] = mask.sum()
df.loc[df["type of election"]=="partisan", "%_complete"] = mean_complete_fed_state
df.loc[df["type of election"]=="partisan", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="partisan", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="partisan", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)
df.loc[df["type of election"]=="partisan", "%_exhausted"] = exh_num[mask].mean(skipna=True)
df.loc[df["type of election"]=="partisan", "first_round_majority"] = is_majority[mask].sum()
df.loc[df["type of election"]=="partisan", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)

# non partisan
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().eq("no")

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="non-partisan", "N"] = mask.sum()
df.loc[df["type of election"]=="non-partisan", "%_complete"] = mean_complete
df.loc[df["type of election"]=="non-partisan", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="non-partisan", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="non-partisan", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)
df.loc[df["type of election"]=="non-partisan", "%_exhausted"] = exh_num[mask].mean(skipna=True)
df.loc[df["type of election"]=="non-partisan", "first_round_majority"] = is_majority[mask].sum()
df.loc[df["type of election"]=="non-partisan", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)


# primary
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().isin(["rp", "dp"])

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="primary", "N"] = mask.sum()
df.loc[df["type of election"]=="primary", "%_complete"] = mean_complete
df.loc[df["type of election"]=="primary", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="primary", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="primary", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)
df.loc[df["type of election"]=="primary", "%_exhausted"] = exh_num[mask].mean(skipna=True)
df.loc[df["type of election"]=="primary", "first_round_majority"] = is_majority[mask].sum()
df.loc[df["type of election"]=="primary", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)

# general
common_files = set(ta["filename"]).intersection(et["filename"])
partisan = ta["filename"].map(et.set_index("filename")["partisan"])
mask = ta["filename"].isin(common_files) & partisan.str.lower().isin(["yes", "no"])

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="general", "N"] = mask.sum()
df.loc[df["type of election"]=="general", "%_complete"] = mean_complete
df.loc[df["type of election"]=="general", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="general", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="general", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)
df.loc[df["type of election"]=="general", "%_exhausted"] = exh_num[mask].mean(skipna=True)
df.loc[df["type of election"]=="general", "first_round_majority"] = is_majority[mask].sum()
df.loc[df["type of election"]=="general", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)


df = df.reset_index(drop=True)
df

KeyError: '%_complete'

In [59]:
# 3 candidate race

mask = ta["candidates"] == 3

df.loc[df["type of election"]=="3 candidate races", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "3 candidate races", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# 4 candidate races

mask = ta["candidates"] == 4

df.loc[df["type of election"]=="4 candidate races", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "4 candidate races", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# 5 candidate races
mask = ta["candidates"] == 5

df.loc[df["type of election"]=="5 candidate races", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "5 candidate races", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# 6 candidate races
mask = ta["candidates"] == 6

df.loc[df["type of election"]=="6 candidate races", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "6 candidate races", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# 7 candidate races
mask = ta["candidates"] == 7

df.loc[df["type of election"]=="7 candidate races", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "7 candidate races", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

# 8 candidate races

mask = ta["candidates"] == 8

df.loc[df["type of election"]=="8 candidate races", "complete_gamma"] = ta.loc[mask, "gamma_complete"].mean(skipna=True)
df.loc[df["type of election"] == "8 candidate races", "%_complete"] = (
    ta.loc[mask, "complete_ballots_count"] / ta.loc[mask, "total_votes"]
).mean(skipna=True)

df


,type of election,N,%_complete,%_bullet,%_truncated,gamma,avg. candidates,%_exhausted,first_round_majority,complete_gamma
0,all elections,361,0.366537,34.303344,NaN,0.817405,5.573407,0.104294,132,0.536824
1,federal/state,57,0.270491,46.694809,NaN,0.884370,4.000000,0.074052,25,0.637701
2,local,304,0.391262,31.979944,NaN,0.804849,5.868421,0.109207,107,0.510856
3,partisan,52,0.233626,50.665682,NaN,0.905509,4.673077,0.089965,23,0.630507
4,non-partisan,244,0.410157,31.336091,NaN,0.826499,5.475410,0.098366,92,0.546921
5,primary,65,0.347574,32.352084,NaN,0.712783,6.661538,0.131723,17,0.359591
6,general,296,0.369669,34.731830,NaN,0.840379,5.334459,0.097020,115,0.566092
7,3 candidate races,125,0.390107,40.962466,NaN,0.899491,3.000000,0.061548,70,0.739125
8,4 candidate races,83,0.383831,36.136461,NaN,0.847514,4.000000,0.071464,33,0.474871
9,5 candidate races,42,0.330493,29.369791,NaN,0.759198,5.000000,0.088524,10,0.177518


In [43]:
df.to_csv("summary_table.csv", index=False)

In [103]:
# 3 candidate race

mask = ta["candidates"] == 3

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="3 candidate races", "N"] = mask.sum()
df.loc[df["type of election"]=="3 candidate races", "%_complete"] = mean_complete
df.loc[df["type of election"]=="3 candidate races", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="3 candidate races", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="3 candidate races", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)

# 4 candidate races

mask = ta["candidates"] == 4

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="4 candidate races", "N"] = mask.sum()
df.loc[df["type of election"]=="4 candidate races", "%_complete"] = mean_complete
df.loc[df["type of election"]=="4 candidate races", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="4 candidate races", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="4 candidate races", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)

# 5 candidate races
mask = ta["candidates"] == 5

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="5 candidate races", "N"] = mask.sum()
df.loc[df["type of election"]=="5 candidate races", "%_complete"] = mean_complete
df.loc[df["type of election"]=="5 candidate races", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="5 candidate races", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="5 candidate races", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)


# 6 candidate races
mask = ta["candidates"] == 6

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="6 candidate races", "N"] = mask.sum()
df.loc[df["type of election"]=="6 candidate races", "%_complete"] = mean_complete
df.loc[df["type of election"]=="6 candidate races", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="6 candidate races", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="6 candidate races", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)

# 7 candidate races
mask = ta["candidates"] == 7

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="7 candidate races", "N"] = mask.sum()
df.loc[df["type of election"]=="7 candidate races", "%_complete"] = mean_complete
df.loc[df["type of election"]=="7 candidate races", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="7 candidate races", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="7 candidate races", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)

# 8 candidate races

mask = ta["candidates"] == 8

mean_complete = ta.loc[mask, "%_complete"].mean(skipna=True)
df.loc[df["type of election"]=="8 candidate races", "N"] = mask.sum()
df.loc[df["type of election"]=="8 candidate races", "%_complete"] = mean_complete
df.loc[df["type of election"]=="8 candidate races", "%_bullet"] = ta.loc[mask, "%_bullet"].mean(skipna=True)
df.loc[df["type of election"]=="8 candidate races", "avg. candidates"] = ta.loc[mask, "candidates"].mean(skipna=True)
df.loc[df["type of election"]=="8 candidate races", "gamma"] = ta.loc[mask, "gamma"].mean(skipna=True)

for k in [3,4,5,6,7,8]:
    mask = ta["candidates"] == k
    label = f"{k} candidate races"
    df.loc[df["type of election"]==label, "%_exhausted"] = exh_num[mask].mean(skipna=True)
    df.loc[df["type of election"]==label, "first_round_majority"] = is_majority[mask].sum()

df


,type of election,N,%_complete,%_bullet,%_truncated,gamma,avg. candidates,%_exhausted,first_round_majority
0,all elections,361.0,39.129348,34.303344,NaN,0.817405,5.573407,0.104294,132.0
1,federal/state,57.0,27.760356,46.694809,NaN,0.884370,4.000000,0.074052,25.0
2,local,304.0,41.261033,31.979944,NaN,0.804849,5.868421,0.109207,107.0
3,partisan,52.0,27.760356,50.665682,NaN,0.905509,4.673077,0.089965,23.0
4,non-partisan,244.0,43.852758,31.336091,NaN,0.826499,5.475410,0.098366,92.0
5,primary,65.0,33.752729,32.352084,NaN,0.712783,6.661538,0.131723,17.0
6,general,296.0,40.310024,34.731830,NaN,0.840379,5.334459,0.097020,115.0
7,3 candidate races,125.0,39.010749,40.962466,NaN,0.899491,3.000000,0.061548,70.0
8,4 candidate races,83.0,38.383081,36.136461,NaN,0.847514,4.000000,0.071464,33.0
9,5 candidate races,42.0,39.741795,29.369791,NaN,0.759198,5.000000,0.088524,10.0


In [45]:
ta.to_csv("truncation_analysis.csv", index=False)

In [15]:
for fn in ta["filename"]:
    filename = fn[:-4] + ".pkl"
    ballots, candidates = load_data(filename, saved)
    e = voting_rules(ballots, candidates)
    total_votes = ta.loc[ta["filename"]==fn, "total_votes"].values[0]
    summary = e.irv()[1]
    # first round had majority
    if set(summary) == {"majority"}:
        ta.loc[ta["filename"]==fn, "%_exhausted"] = "majority"
    else:
        exhausted_votes = total_votes -  sum(summary["majority"].values())
        ta.loc[ta["filename"]==fn, "%_exhausted"] = exhausted_votes / total_votes

ta
    

,Unnamed: 0,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,...,voluntarily_truncated_votes,total_votes,level,%_bullet,%_truncated,all_candidates_listed,all_choices_filled,complete_ballot (either case),%_complete,%_exhausted
0,0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,...,NaN,19742,FEDERAL,19.866275,NaN,NaN,5344.0,5344.0,27.069193,majority
1,1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick",...,133333.0,188852,FEDERAL,29.681444,70.601847,55519.0,NaN,55519.0,29.398153,0.059348
2,2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,...,301747.0,338650,FEDERAL,67.408239,89.102909,36903.0,NaN,36903.0,10.897091,majority
3,3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.",...,6769.0,8163,STATE,65.049614,82.922945,1394.0,NaN,1394.0,17.077055,majority
4,4,Alaska_11052024_StateHouseD15.csv,3,4,0.982540,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny",...,7576.0,8820,STATE,70.714286,85.895692,1244.0,NaN,1244.0,14.104308,majority
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,...,385.0,1537,LOCAL,20.624593,25.048796,1152.0,NaN,1152.0,74.951204,majority
357,357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,...,452.0,1089,LOCAL,5.876951,41.505969,637.0,NaN,637.0,58.494031,0.040404
358,358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",...,5663.0,9560,LOCAL,46.694561,59.236402,3897.0,NaN,3897.0,40.763598,0.116946
359,359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,...,262.0,655,LOCAL,19.541985,40.000000,393.0,NaN,393.0,60.000000,0.085496


In [7]:
df = pd.read_csv("summary_table.csv")
df

,type of election,N,%_complete,%_bullet,%_truncated,gamma,avg. candidates,%_exhausted,first_round_majority
0,all elections,361,39.129348,34.303344,NaN,0.817405,5.573407,0.104294,132
1,federal/state,57,27.760356,46.694809,NaN,0.884370,4.000000,0.074052,25
2,local,304,41.261033,31.979944,NaN,0.804849,5.868421,0.109207,107
3,partisan,52,27.760356,50.665682,NaN,0.905509,4.673077,0.089965,23
4,non-partisan,244,43.852758,31.336091,NaN,0.826499,5.475410,0.098366,92
5,primary,65,33.752729,32.352084,NaN,0.712783,6.661538,0.131723,17
6,general,296,40.310024,34.731830,NaN,0.840379,5.334459,0.097020,115
7,3 candidate races,125,39.010749,40.962466,NaN,0.899491,3.000000,0.061548,70
8,4 candidate races,83,38.383081,36.136461,NaN,0.847514,4.000000,0.071464,33
9,5 candidate races,42,39.741795,29.369791,NaN,0.759198,5.000000,0.088524,10


In [4]:
et = pd.read_csv("election_table.csv")
et

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,filename,null_gamma,null_gamma_recalculated,complete_gamma,gamma,choices,candidates,...,median_voter_preference_position,median_voter_position,irv_median_voter_distance,condorcet_median_voter_distance,median_voter_preference_distance,bimodality,partisan2,diff,linearvotersfailure,mirror
0,454,454,92,NewYorkCity_06222021_CONCouncilMember19thCounc...,1.000000,1.000000,NaN,1.000000,3,2,...,0.000000,0.443536,0.556464,0.556464,0.000000,4.246498,NaN,[],no,NaN
1,5,10,495,NewYorkCity_06222021_DEMCouncilMember27thCounc...,0.136963,0.127073,0.222547,0.488639,5,12,...,0.391163,0.443536,0.054424,0.054424,0.009847,2.544507,NaN,[],no,NaN
2,7,7,503,NewYorkCity_06222021_DEMCouncilMember26thCounc...,0.119381,0.119481,0.235185,0.480000,5,15,...,0.468333,0.443536,0.069082,0.069082,0.007974,2.692441,NaN,[],no,NaN
3,12,25,491,NewYorkCity_06222021_DEMCouncilMember40thCounc...,0.136064,0.134665,0.251027,0.602433,5,11,...,0.425919,0.443536,0.212378,0.212378,0.068476,2.587658,NaN,[],no,NaN
4,15,49,378,NewYorkCity_06222021_DEMBoroughPresidentRichmo...,0.325100,0.258000,0.261159,0.685076,5,5,...,0.543824,0.443536,0.100288,0.100288,0.000000,2.974382,NaN,[],no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
510,428,428,66,Alaska_11082022_SenateDistrictO.csv,1.000000,1.000000,NaN,1.000000,3,2,...,0.000000,0.443536,0.443536,0.443536,0.000000,5.147279,1.0,[],no,NaN
511,429,429,67,Alaska_11082022_SenateDistrictS.csv,1.000000,1.000000,NaN,1.000000,3,2,...,0.000000,0.443536,0.443536,0.443536,0.000000,5.057274,1.0,[],no,NaN
512,440,440,78,Burlington_03072023_CityCouncilCentralDistrict...,1.000000,1.000000,NaN,1.000000,3,2,...,0.000000,0.443536,0.556464,0.556464,0.000000,4.615537,1.0,[],no,NaN
513,441,441,79,Burlington_03072023_CityCouncilEastDistrict.csv,1.000000,1.000000,NaN,1.000000,3,2,...,0.000000,0.443536,0.556464,0.556464,0.000000,4.603198,1.0,[],no,NaN


In [3]:
saved = "saved_ballots_and_candidates"
count = 0
# adding a "complete ballot" = all of the choices on the ballot are filled out 
# adding a "all candidates listed" = choices on the ballot = number of candidates 

ta["all_candidates_listed"] = np.nan
ta["all_choices_filled"] = np.nan

for filename in ta["filename"]:
    fn =  filename[:-4]+ ".pkl"
    ballots, candidates = load_data(fn, saved)
    if len(candidates) > 2:
        candidates, choices = ta.loc[ta["filename"]==filename, "candidates"].values[0], ta.loc[ta["filename"]==filename, "choices"].values[0]
        if candidates <= choices:
            # second condition
            complete_ballot_counter = 0
            for b in ballots:
                if len(b) == candidates:
                    complete_ballot_counter += ballots[b]
            ta.loc[ta["filename"]==filename , "all_candidates_listed"] = complete_ballot_counter
            

        if candidates > choices:
            # first condition
            complete_ballot_counter = 0
            for b in ballots:
                if len(b) == choices:
                    complete_ballot_counter += ballots[b]
            ta.loc[ta["filename"]==filename, "all_choices_filled"] = complete_ballot_counter
            
        ta.loc[ta["filename"]==filename, "complete_ballot (either case)"] = complete_ballot_counter
        ta["%_complete"] = 100 * ta["complete_ballot (either case)"] / ta["total_votes"]
        
ta




NameError: name 'ta' is not defined

In [9]:
ta = pd.read_csv("truncation_analysis.csv")
ta

,Unnamed: 0,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level
0,0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,NaN,NaN,NaN,3922,NaN,19742,FEDERAL
1,1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333.0,188852,FEDERAL
2,2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747.0,338650,FEDERAL
3,3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769.0,8163,STATE
4,4,Alaska_11052024_StateHouseD15.csv,3,4,0.982540,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576.0,8820,STATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385.0,1537,LOCAL
357,357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452.0,1089,LOCAL
358,358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663.0,9560,LOCAL
359,359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262.0,655,LOCAL


In [10]:
from consistency import*

In [46]:
def get_gamma_for_complete_ballots(filename, ta):
    saved = "saved_ballots_and_candidates"
    #filename = "Berkeley_11052024_Mayor.csv"
    candidates, choices = ta.loc[ta["filename"]==filename, "candidates"].values[0], ta.loc[ta["filename"]==filename, "choices"].values[0]
    ballots, candidate_names = load_data(filename[:-4]+".pkl", saved)

    if candidates > choices + 1:
        return None, None
    
    for ballot in list(ballots):
        # number of candidats is the same as the number of choices on the ballot
        if candidates <= choices:
            if len(ballot) < candidates:
                del ballots[ballot]
        if candidates == choices + 1:
            # if there is only one candidate missing from the ballot, add it and count it as a complete ballot
            if len(ballot) == candidates - 1:
                for c in candidate_names:
                    if c not in ballot:
                        ballot_altered = ballot + (c, )
                        if ballot_altered in ballots:
                            ballots[ballot_altered] += ballots[ballot]
                        else:
                            ballots[ballot_altered] = ballots[ballot]
                        del ballots[ballot]
                        
            # else, remove the ballot (shouldn't enter here)
            else:
                del ballots[ballot]



    fn = "null_elections/" + filename

    meta = pd.read_csv(fn, usecols=["candidate", "position"])
    meta = meta.dropna(subset=["candidate", "position"])
    meta["candidate"] = meta["candidate"].astype(str).str.strip()

    pmin, pmax = meta["position"].min(), meta["position"].max()
    den = pmax - pmin
    meta["position_norm"] = (meta["position"] - pmin) / den if den > 0 else 0.0

    # if duplicates exist, keep the first row per candidate
    normalized_distances = (
        meta.drop_duplicates("candidate", keep="first")
        .set_index("candidate")["position_norm"]
        .to_dict()
    )
    complete_ballots_count = sum(ballots.values())
    return get_permissive_gamma(ballots, normalized_distances)[1], complete_ballots_count


In [49]:
filename = "Berkeley_11042014_CityCouncilDistrict8.csv"
gamma_complete, complete_ballot_counter = get_gamma_for_complete_ballots(filename, ta)
print(gamma_complete)
print(complete_ballot_counter)

0.3692201518288475
2898


In [50]:
for filename in ta["filename"]:
    try:
        gamma_complete, complete_ballot_counter = get_gamma_for_complete_ballots(filename, ta)
        ta.loc[ta["filename"]==filename, "gamma_complete"] = gamma_complete
        ta.loc[ta["filename"]==filename, "complete_ballots_count"] = complete_ballot_counter
    except Exception as e:
        print(filename, " ", e)
ta

TakomaPark_11082022_CityCouncilWard3.csv   division by zero


,Unnamed: 0,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level,gamma_complete,complete_ballots_count
0,0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,NaN,NaN,NaN,3922,NaN,19742,FEDERAL,NaN,NaN
1,1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333.0,188852,FEDERAL,0.887210,55519.0
2,2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747.0,338650,FEDERAL,0.000108,36903.0
3,3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769.0,8163,STATE,0.729555,1394.0
4,4,Alaska_11052024_StateHouseD15.csv,3,4,0.982540,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576.0,8820,STATE,0.876206,1244.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385.0,1537,LOCAL,0.734375,1152.0
357,357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452.0,1089,LOCAL,0.004710,637.0
358,358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663.0,9560,LOCAL,0.730562,3897.0
359,359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262.0,655,LOCAL,0.770992,393.0


In [51]:
ta.to_csv("truncation_analysis.csv", index=False)

In [81]:
test = perform_rcv_analysis(ballots, candidate_names, n_init = 100, max_itr = 10000, n_runs=1000, metric=False)
mds_1d_coordinates, mds_2d_coordinates, most_common_order, order_frequencies, candidate_names = test

# Print the normalized distances between candidates and plot the MDS analysis
normalized_distances_complete = get_distances_normalized(most_common_order, mds_1d_coordinates, candidate_names)
print(normalized_distances_complete)
consistent_ballots, gamma = get_permissive_gamma(ballots, normalized_distances_complete)


c:\Users\ual-laptop\Desktop\RCV\bugs-in-democracy\cleaned files\MDS_analysis.py:162: RuntimeWarning: divide by zero encountered in divide
  distance = 1 / np.sqrt(freq_upper_triangle)
c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will 

{'Cesar A. Vargas': np.float64(0.0), 'Radhakrishna Mohan': np.float64(0.0032484262414316134), 'Brandon P. Stradford': np.float64(1.3755404708609353), 'Lorraine A. Honor': np.float64(2.6274722521628355), 'Mark S. Murphy': np.float64(4.0)}


c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\ual-laptop\anaconda3\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(


In [82]:
print(gamma)
print(normalized_distances_complete)

0.08144342814183686
{'Cesar A. Vargas': np.float64(0.0), 'Radhakrishna Mohan': np.float64(0.0032484262414316134), 'Brandon P. Stradford': np.float64(1.3755404708609353), 'Lorraine A. Honor': np.float64(2.6274722521628355), 'Mark S. Murphy': np.float64(4.0)}


In [ ]:
saved = "saved_ballots_and_candidates"

for filename in ta["filename"]:
    
    candidates, choices = ta.loc[ta["filename"]==filename, "candidates"].values[0], ta.loc[ta["filename"]==filename, "choices"].values[0]
    ballots, candidate_names = load_data(filename[:-4]+".pkl, saved")

    for ballot in ballots:
        # number of candidats is the same as the number of choices on the ballot
        if candidates <= choices:
            for ballot in ballots:
                if len(ballot) < candidates:
                    del ballots[ballot]
        if candidates == choices + 1:
            for ballot in ballots:
                # if there is only one candidate missing from the ballot, add it and count it as a complete ballot
                if len(ballot) == candidates - 1:
                    for c in candidate_names:
                        if c not in ballot:
                            ballot += (c, )
                # else, remove the ballot
                else:
                    del ballots[ballot]



    fn = "null_elections/" + filename

    meta = pd.read_csv(fn, usecols=["candidate", "position"])
    meta = meta.dropna(subset=["candidate", "position"])
    meta["candidate"] = meta["candidate"].astype(str).str.strip()

    pmin, pmax = meta["position"].min(), meta["position"].max()
    den = pmax - pmin
    meta["position_norm"] = (meta["position"] - pmin) / den if den > 0 else 0.0

    # if duplicates exist, keep the first row per candidate
    normalized_distances = (
        meta.drop_duplicates("candidate", keep="first")
        .set_index("candidate")["position_norm"]
        .to_dict()
    )



IndentationError: expected an indented block after 'for' statement on line 6 (514939289.py, line 8)